# OpenAlex Topics Embeddings

In [37]:
import pandas as pd
from IPython.core.display import JSON

In [7]:
xl = pd.ExcelFile("Copy of OpenAlex_topic_mapping_table_new.xlsx")
xl.sheet_names

['final_topic_field_subfield_tabl', 'Sheet1']

In [8]:
df = xl.parse("final_topic_field_subfield_tabl")
df.head()

,topic_id,old_topic_label,new_topic_label,subfield_id,subfield_name,field_id,field_name,domain_id,domain_name,keywords,summary,wikipedia_url
0,10001,Tectonic and Geochronological Evolution of Oro...,Geological and Geochemical Analysis,1908,Geophysics,19,Earth and Planetary Sciences,3,Physical Sciences,Zircon; Geochronology; Tectonics; Granitic Roc...,This cluster of papers focuses on the tectonic...,https://en.wikipedia.org/wiki/Geochronology
1,10002,Advancements in Density Functional Theory,Advanced Chemical Physics Studies,3107,"Atomic and Molecular Physics, and Optics",31,Physics and Astronomy,3,Physical Sciences,Density Functional Theory; Dispersion Correcti...,This cluster of papers represents advancements...,https://en.wikipedia.org/wiki/Density_function...
2,10003,Knowledge Management and Organizational Innova...,Innovation and Knowledge Management,1408,Strategy and Management,14,"Business, Management and Accounting",2,Social Sciences,Dynamic Capabilities; Knowledge Transfer; Busi...,This cluster of papers revolves around the top...,https://en.wikipedia.org/wiki/Knowledge_manage...
3,10004,Soil Carbon Dynamics and Nutrient Cycling in E...,Soil Carbon and Nitrogen Dynamics,1111,Soil Science,11,Agricultural and Biological Sciences,1,Life Sciences,Soil Carbon Sequestration; Nitrogen Cycle; Mic...,This cluster of papers explores the dynamics o...,https://en.wikipedia.org/wiki/Soil_carbon_dyna...
4,10005,Biodiversity Conservation and Ecosystem Manage...,Ecology and Vegetation Dynamics Studies,2309,Nature and Landscape Conservation,23,Environmental Science,3,Physical Sciences,Biodiversity; Conservation; Ecosystem; Invasiv...,This cluster of papers focuses on the conserva...,https://en.wikipedia.org/wiki/Biodiversity_con...


# Creating the structure

In [55]:
attributes = ('topics','keywords','old_topics','broaders','narrowers','all_broaders','level')

openalex = {}
for attr in attributes:
    openalex[attr] = {}

for idx, row in df.iterrows():
    # topic
    openalex['topics'][row['new_topic_label']] = row['topic_id']
    openalex['topics'][row['subfield_name']] = row['subfield_id']
    openalex['topics'][row['field_name']] = row['field_id']
    openalex['topics'][row['domain_name']] = row['domain_id']

    # keywords
    t_keywords = [keyword.strip() for keyword in row['keywords'].split(';')]
    for keyword in t_keywords:
        if keyword not in openalex['keywords']:
            openalex['keywords'][keyword] = list()
        openalex['keywords'][keyword].append(row['new_topic_label'])

    # old_topics
    openalex['old_topics'][row['old_topic_label']] = row['topic_id']

    # broaders
    if row['new_topic_label'] not in openalex['broaders']:
        openalex['broaders'][row['new_topic_label']] = list()
    openalex['broaders'][row['new_topic_label']].append(row['subfield_name'])
    if row['subfield_name'] not in openalex['broaders']:
        openalex['broaders'][row['subfield_name']] = list()
    openalex['broaders'][row['subfield_name']].append(row['field_name'])
    if row['field_name'] not in openalex['broaders']:
        openalex['broaders'][row['field_name']] = list()
    openalex['broaders'][row['field_name']].append(row['domain_name'])
    
    # narrowers
    if row['domain_name'] not in openalex['narrowers']:
        openalex['narrowers'][row['domain_name']] = list()
    openalex['narrowers'][row['domain_name']].append(row['field_name'])
    if row['field_name'] not in openalex['narrowers']:
        openalex['narrowers'][row['field_name']] = list()
    openalex['narrowers'][row['field_name']].append(row['subfield_name'])
    if row['subfield_name'] not in openalex['narrowers']:
        openalex['narrowers'][row['subfield_name']] = list()
    openalex['narrowers'][row['subfield_name']].append(row['new_topic_label'])
    
    # all_broaders
    for keyword in t_keywords:
        if keyword not in openalex['all_broaders']:
            openalex['all_broaders'][keyword] = []
        openalex['all_broaders'][keyword] += [row['new_topic_label'], row['subfield_name'], row['field_name'], row['domain_name']]
    if row['old_topic_label'] not in openalex['all_broaders']:
        openalex['all_broaders'][row['old_topic_label']] = [row['subfield_name'], row['field_name'], row['domain_name']]
    if row['new_topic_label'] not in openalex['all_broaders']:
        openalex['all_broaders'][row['new_topic_label']] = [row['subfield_name'], row['field_name'], row['domain_name']]
    if row['subfield_name'] not in openalex['all_broaders']:
        openalex['all_broaders'][row['subfield_name']] =  [row['field_name'], row['domain_name']]
    if row['field_name'] not in openalex['all_broaders']:
        openalex['all_broaders'][row['field_name']]  = [row['domain_name']]
    
    # level
    openalex['level'][row['new_topic_label']] = 3
    openalex['level'][row['subfield_name']] = 2
    openalex['level'][row['field_name']] = 1
    openalex['level'][row['domain_name']] = 0


# clean broaders and narrower

for k,v in openalex['broaders'].items():
    openalex['broaders'][k] = list(set(v))
for k,v in openalex['narrowers'].items():
    openalex['narrowers'][k] = list(set(v))
for k,v in openalex['all_broaders'].items():
    openalex['all_broaders'][k] = list(set(v))

In [56]:
#JSON(openalex)

<IPython.core.display.JSON object>

# Extracting the embeddings

In [35]:
unique_concepts = set(df['new_topic_label']).union(set(df['subfield_name'])).union(set(df['field_name'])).union(set(df['domain_name'])).union(set(df['old_topic_label']))
len(unique_concepts)

9231

In [31]:
list_of_keywords = []
for idx, row in df.iterrows():
    list_of_keywords += [keyword.strip() for keyword in row['keywords'].split(';')]
print(len(list_of_keywords))
print(len(set(list_of_keywords)))

45154
27855


In [36]:
unique_concepts = unique_concepts.union(set(list_of_keywords))
len(unique_concepts)

36941

In [47]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = list(unique_concepts)

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

vector_size = embeddings.shape[1]

/opt/anaconda3/envs/dev38/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


(36941, 384)


In [48]:
import faiss                   # make faiss available
index = faiss.IndexFlatL2(vector_size)   # build the index
print(index.is_trained)

True


In [49]:
index.add(embeddings)                  # add vectors to the index
print(index.ntotal)

36941


In [57]:
to_save = dict()
to_save["sentences"] = sentences
to_save["index"] = index
to_save["structure"] = openalex 
import pickle
with open('openalex.pickle', 'wb') as handle:
    pickle.dump(to_save, handle, protocol=pickle.HIGHEST_PROTOCOL)